# 12.7 - RAG With LangChain
**Phase:** 12 - LangChain / Framework Abstractions
**Status:** VERIFIED
---
## 1. What Are We Solving?
Phase 11 built RAG by hand. LangChain provides ready-made components: splitters, an embeddings
wrapper, a Chroma vector store, retrievers, and chain composition. This unit maps the manual
patterns onto framework abstractions.
## 2. Why Does This Matter?
RAG is the most common real-world LLM pattern. LangChain standardises load -> split -> embed ->
store -> retrieve -> generate into a few lines that are easy to reason about and swap.
## 3. Prerequisites
- Phase 11 (RAG fundamentals)
- Units 12.1-12.6
## 4. Learning Objectives
By the end of this unit, you should be able to:
- Split documents with `RecursiveCharacterTextSplitter`
- Build Chroma via `Chroma.from_documents(splits, embeddings)`
- Build a retriever with `.as_retriever(search_kwargs={"k": 4})`
- Compose a grounded RAG chain over `|` with `StrOutputParser`
- (Optional) recognise legacy `RetrievalQA.from_llm`; print sources
## 5. Mental Model
RAG is an assembly line: documents go in, get chopped into chunks, embedded into vectors, stored in
Chroma, retrieved per query, injected into a prompt, and answered by the model.

```text
docs -> split -> embed -> Chroma -> as_retriever(k=4) -> retrieve -> format -> prompt -> model -> answer
                                                                                          \-> print sources


## 6. Setup + LLM helper

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


In [2]:
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


class MiniLM(Embeddings):
    """sentence-transformers wrapper with a deterministic hash fallback (offline-safe)."""
    def __init__(self):
        self.ok = False
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.ok = True
        except Exception:
            self.model = None

    def _vec(self, text: str):
        if self.ok:
            try:
                return self.model.encode([text], convert_to_numpy=True)[0].tolist()
            except Exception:
                pass
        import hashlib
        dim = 32
        v = [0.0] * dim
        for ch in text.lower():
            h = int(hashlib.md5(ch.encode()).hexdigest(), 16)
            v[h % dim] += 1.0
        s = sum(x * x for x in v) ** 0.5 or 1.0
        return [round(x / s, 6) for x in v]

    def embed_documents(self, texts):
        return [self._vec(t) for t in texts]

    def embed_query(self, text):
        return self._vec(text)


print("embeddings wrapper ready (model loaded:", MiniLM().ok, ")")


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\PC\AppData\Local\Temp\ipykernel_3028\119774918.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2241.29it/s]

embeddings wrapper ready (model loaded: True )


## 7. Load Text Into Documents and Split
We do not need files: build a small FAQ as `Document` objects and split them into chunks
(`chunk_size` + `chunk_overlap` keep context across boundaries).

In [3]:
docs = [
    Document(page_content="Our refund policy allows a full refund within 30 days of purchase. "
                          "Returned items must be in original packaging."),
    Document(page_content="Free shipping applies to orders over 50 dollars. Standard delivery takes "
                          "2-4 business days; express delivery arrives in 1-2 days."),
    Document(page_content="Canceling a subscription keeps access until the end of the current billing "
                          "period, then stops auto-renewal."),
    Document(page_content="You can change your password from Settings > Security. You will receive a "
                          "confirmation email after the change."),
    Document(page_content="The mobile app supports both iOS and Android. Offline mode stores up to "
                          "200 documents for reading without a connection."),
    Document(page_content="Our support team is available 24/7 by chat and email. Average first "
                          "response time is under 15 minutes."),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
splits = splitter.split_documents(docs)
print("docs:", len(docs), "-> chunks:", len(splits))
for i, c in enumerate(splits[:4]):
    print(f"  chunk {i}: {c.page_content[:45]}...")


docs: 6 -> chunks: 7
  chunk 0: Our refund policy allows a full refund within...
  chunk 1: Free shipping applies to orders over 50 dolla...
  chunk 2: delivery arrives in 1-2 days....
  chunk 3: Canceling a subscription keeps access until t...


## 8. Embed + Store in Chroma, Build a Retriever
`Chroma.from_documents(splits, embeddings)` creates an ephemeral in-memory store. `.as_retriever`
with `k=4` returns the top-4 most similar chunks per query.

In [4]:
retriever = None
store = None
try:
    store = Chroma.from_documents(splits, MiniLM())
    retriever = store.as_retriever(search_kwargs={"k": 4})
    print("Chroma store built; retriever ready with k=4")
except Exception as e:
    print("Chroma build fell back (see next cell for manual retrieval):", type(e).__name__)


def retrieve(query: str):
    if retriever is not None:
        return retriever.invoke(query)
    # manual fallback: cosine over hash embeddings (no framework)
    em = MiniLM()
    qv = em.embed_query(query)
    scored = []
    for c in splits:
        dv = em.embed_query(c.page_content)
        sim = sum(a * b for a, b in zip(qv, dv))
        scored.append((sim, c))
    scored.sort(key=lambda t: -t[0])
    return [c for _, c in scored[:4]]


sample = retrieve("refund within how many days?")
for d in sample:
    print("-", d.page_content[:70])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3164.54it/s]

Chroma store built; retriever ready with k=4


- Our refund policy allows a full refund within 30 days of purchase. Ret
- Our support team is available 24/7 by chat and email. Average first re
- delivery arrives in 1-2 days.
- Free shipping applies to orders over 50 dollars. Standard delivery tak


## 9. The Grounded RAG Chain
Compose `context-gen` (retriever + format) and `question` into the prompt, then `model -> parser`.
The prompt tells the model to answer only from context and to admit when it cannot — this is what
grounds the answer and fights hallucination.

In [5]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


rag_prompt = ChatPromptTemplate.from_template(
    "Answer ONLY from the context below. If the context does not contain the answer, say "
    "'I don't know based on the provided context.'\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

from langchain_core.runnables import RunnableLambda, RunnablePassthrough


def safe_model(messages):
    if not os.environ.get("GROQ_API_KEY"):
        return "mock grounded answer: based on the retrieved context."
    try:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, temperature=0.0).invoke(messages).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


# Show the retriever | format_docs shorthand that feeds the prompt's {context}.
retriever_demo = lambda q: format_docs(retrieve(q))
print("reformatted context (first 80 chars):", retriever_demo("refund policy")[:80])

get_ctx = RunnableLambda(lambda q: format_docs(retrieve(q)))
rag_chain = (
    {"context": get_ctx, "question": RunnablePassthrough()}
    | rag_prompt
    | RunnableLambda(lambda msgs: safe_model(msgs))
    | StrOutputParser()
)
print("rag_chain composed")


reformatted context (first 80 chars): Our refund policy allows a full refund within 30 days of purchase. Returned item
rag_chain composed


In [6]:
def safe_chat(messages):
    if not os.environ.get("GROQ_API_KEY"):
        return "mock grounded answer: based on the retrieved context."
    try:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, temperature=0.0).invoke(messages).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


rag_chain2 = (
    {"context": get_ctx, "question": RunnablePassthrough()}
    | rag_prompt
    | RunnableLambda(lambda msgs: safe_chat(msgs))
    | StrOutputParser()
)


def ask(q: str):
    ctx_docs = retrieve(q)
    answer = rag_chain2.invoke(q)
    print("Q:", q)
    print("A:", answer)
    print("Sources:")
    for i, d in enumerate(ctx_docs, 1):
        print(f"  [{i}] {d.page_content[:60]}...")
    print()


ask("How long do I have to ask for a refund?")
ask("What is express delivery time?")
ask("Do you sell furniture?")   # not in docs -> should say I don't know


Q: How long do I have to ask for a refund?
A: You have 30 days from the date of purchase to request a refund.
Sources:
  [1] Our refund policy allows a full refund within 30 days of pur...
  [2] Our support team is available 24/7 by chat and email. Averag...
  [3] Free shipping applies to orders over 50 dollars. Standard de...
  [4] Canceling a subscription keeps access until the end of the c...



Q: What is express delivery time?
A: I don't know based on the provided context.
Sources:
  [1] delivery arrives in 1-2 days....
  [2] Free shipping applies to orders over 50 dollars. Standard de...
  [3] Our support team is available 24/7 by chat and email. Averag...
  [4] Our refund policy allows a full refund within 30 days of pur...



Q: Do you sell furniture?
A: I don't know based on the provided context.
Sources:
  [1] delivery arrives in 1-2 days....
  [2] Free shipping applies to orders over 50 dollars. Standard de...
  [3] Our refund policy allows a full refund within 30 days of pur...
  [4] Canceling a subscription keeps access until the end of the c...



## 10. (Optional) Legacy RetrievalQA.from_llm
`RetrievalQA` from `langchain.chains` is the old high-level RAG builder. It still works but is
**legacy** — the `|` chain above gives you more control. We show it only for recognition.

In [7]:
try:
    from langchain.chains import RetrievalQA
    from langchain_groq import ChatGroq
    qa = RetrievalQA.from_llm(llm=ChatGroq(model=GROQ_MODEL, temperature=0.0),
                              retriever=retriever if retriever is not None else None)
    print("legacy RetrievalQA constructed. Prefer the custom `|` chain for control.")
except Exception as e:
    print("RetrievalQA (legacy) not built:", type(e).__name__)


RetrievalQA (legacy) not built: ModuleNotFoundError




## Common Mistakes

- (3-5 bullets, from roadmap, concrete and specific to the unit)

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| ... | ... | ... |
(row table, 3-5 rows)

## Best Practices

- (3-5 bullets)

## Hands-On Practice

1. **Basic:** ...
2. **Guided:** ...
3. **Independent:** ...
4. **Realistic:** ...
5. **Challenge:** ...

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.


### Common Mistakes (applied)

- Chunks too big (irrelevant context) or too small (lost context); no overlap.
- Retrieving too few/many chunks; poor `k`.
- No grounding instruction -> hallucinated answers.
- Ignoring sources when debugging answer quality.

### Debugging (applied)

| Symptom | Likely Cause | Fix |
|---|---|---|
| Answer irrelevant | Retrieved chunks irrelevant | Inspect `retrieve(q)`; tune chunk size |
| Answer partial | Chunks miss full answer | Raise `chunk_size` / lower `k` |
| Hallucinated answer | No grounding instruction | Add "answer only from context" |
| No answer | Empty retrieval | Check documents were split/embedded |

### Best Practices (applied)

- Chunk by semantic units, use overlap.
- Add metadata (source) for filtering.
- Evaluate retrieval quality separately from answer quality.

### Hands-On Practice

1. **Basic:** Ask 3 new questions and print their sources.
2. **Guided:** Tune `chunk_size`/`overlap` and note answer changes.
3. **Independent:** Add an 8th FAQ document and re-query.
4. **Realistic:** Add metadata (source tag) and filter on it during retrieval.
5. **Challenge:** Compare 2-3 retrieval `k` values on 10 questions and score answer quality.

### Exit Criteria

- You can build a full RAG pipeline with LangChain.
- You can choose appropriate chunking and retrieval parameters.
- You can debug retrieval and answer quality issues.
